# 03 Quantity intent: counting animal legs

**Question.** Does the quantity intent of the SDK count the legs of an animal correctly, on the
original image and on the image edited to have one more leg?

**Why.** Every vision-language model that VLMBias tested gives the normal number of legs for the
species on the edited image. A count in code from the image must see the extra leg.

**Approach.** `scripts/run_legs.py` runs `Harness.answer` with the tools and intents of
`.env.paper`, a fixed classifier that chooses `quantity`, and the question "How many legs does
this animal have?". The selector chooses the legs counter. It finds leg regions with SAM 3 and the
concept "leg", removes duplicate regions, and counts in code. No language model writes the count.
The images: the 1,098 pairs of the VLMBias linear-probing split (`train-animals.parquet`), each an
original and its edited image at the same size.

**Prediction.** More than 80% of originals and of edited images counted correctly. On most pairs
the edited count is higher than the original count.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import runs

RECORDED = runs.recorded_dir()
if RECORDED is None:
    print("INTENT_HARNESS_RECORDED_DIR is not set: the comparison with earlier results is skipped.")
else:
    print("INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.")

INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.


In [2]:
table = runs.read_result("03-legs")
table["correct"] = table["count"] == table["ground_truth"]
originals = table[table["image_type"] == "original"]
edited = table[table["image_type"] == "edited"]
pairs = originals.merge(edited, on=["base", "pixel"], suffixes=("_original", "_edited"))
print("images:", len(table), "| pairs:", len(pairs))

images: 2196 | pairs: 1098


## 1. Leg counts on 1,098 pairs

A count is correct when it equals the species' normal number of legs on the original, and that
number plus one on the edited image. Each pair also compares the edited count with the original
count, correct or not: a higher edited count means the counter saw the added leg. The sign test
asks whether "higher" is more frequent than "lower".

In [3]:
from scipy.stats import binomtest

higher = int((pairs["count_edited"] > pairs["count_original"]).sum())
same = int((pairs["count_edited"] == pairs["count_original"]).sum())
lower = int((pairs["count_edited"] < pairs["count_original"]).sum())
clean = int((pairs["correct_original"] & pairs["correct_edited"]).sum())
pd.DataFrame([{
    "pairs": len(pairs),
    "originals correct": round(originals["correct"].mean(), 3),
    "edited correct": round(edited["correct"].mean(), 3),
    "both correct (pairs)": clean,
    "edited count higher": higher,
    "edited count same": same,
    "edited count lower": lower,
    "sign test p": binomtest(higher, higher + lower).pvalue,
}])

,pairs,originals correct,edited correct,both correct (pairs),edited count higher,edited count same,edited count lower,sign test p
0,1098,0.856,0.839,822,985,100,13,1.094419e-271


## 2. By image size

In [4]:
rows = []
for pixel, part in table.groupby("pixel"):
    rows.append({
        "pixel": pixel,
        "originals correct": round(part[part["image_type"] == "original"]["correct"].mean(), 3),
        "edited correct": round(part[part["image_type"] == "edited"]["correct"].mean(), 3),
        "seconds per image": round(part["seconds"].mean(), 2),
    })
pd.DataFrame(rows)

,pixel,originals correct,edited correct,seconds per image
0,384,0.852,0.855,1.01
1,768,0.866,0.836,1.04
2,1152,0.850,0.825,1.24


## 3. Comparison with the earlier results, item by item

Runs only when `INTENT_HARNESS_RECORDED_DIR` is set: the count of each image against the earlier
SAM 3 result (`nms+nested`).

In [5]:
if RECORDED is not None:
    earlier = pd.read_parquet(RECORDED / "07-full-counts.parquet")
    earlier = earlier[(earlier["arm"] == "sam3") & (earlier["merge"] == "nms+nested")]
    both = table.merge(earlier[["sample", "count"]], on="sample", suffixes=("", "_earlier"))
    print("images compared:", len(both))
    print("same count:", int((both["count"] == both["count_earlier"]).sum()))
    different = both[both["count"] != both["count_earlier"]]
    if len(different):
        display(different[["sample", "name", "pixel", "image_type", "ground_truth", "count", "count_earlier"]])

images compared: 2196
same count: 2196


## Reading the result

The legs counter gives the correct count on 0.856 of the originals and on 0.839 of the edited
images. Both counts are correct on 822 of 1,098 pairs. The prediction (more than 80% of each) is
true.

The edited count is higher than the original count on 985 pairs, the same on 100 pairs and lower
on 13 pairs. The counter sees the added leg on most pairs, also on pairs where one of its counts
is wrong. The sign test gives p = 1.1e-271.

The image size has a small effect: from 384 px to 1152 px the accuracy on edited images goes from
0.855 to 0.825.

The comparison with the earlier SAM 3 results gives the same count on 2,196 of 2,196 images.

## Conclusion

The quantity intent of the SDK counts legs in code from the image and sees the added leg on 985
of 1,098 pairs. It reproduces the earlier SAM 3 counts exactly.